Instru Testing Signals from Build / Config files

In [ ]:
# -*- coding: utf-8 -*-
"""
Unified CI + GMD config analysis.

Part 1 (unchanged detection logic):
- Scan CI YAML files for instrumentation-testing signals and output:
  filename, full_name, ci_platform, instru_t_ci_signal,
  execution_environment, test_invocation, flutter_integ_t_signal, flutter_integ_t_d

Part 2 (unchanged GMD logic):
- Scan ALL Gradle scripts (*.gradle, *.gradle.kts) for Managed Devices (GMD) blocks
  and extract device_profile/api_level/image_source/device_identifier + spans.

Unified output (one row per config / build file):
- filename
- full_name
- ci_platform
- instru_t_ci_signal
- execution_environment
- test_invocation
- flutter_integ_t_signal
- flutter_integ_t_d
- Exec_Env_Style                  (Emu_Community, Emu_Custom, GMD, Third-Party, Real Device)
- Test_Invoc_Style                (Gradle-based, Third-Party CLI, ADB)
- device_profile                  (GMD details for Gradle files; blank for YAML)
- api_level
- image_source
- device_identifier
- context_anchor
- span_start_line
- span_end_line
"""

import os
import re
import json
import pandas as pd
from pathlib import Path
from typing import List, Pattern, Tuple, Dict, Any, Iterable, Set, Optional

# === CONFIG ===
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files")
OUTPUT_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\1_All_Configs_Instru.csv")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# === Shared helpers ===
def extract_full_name_from_file(filename: str) -> str:
    fname = filename.lower()
    if "__" in fname:
        return fname.split("__", 1)[0]
    return Path(fname).stem

def extract_ci_platform(filename: str) -> str:
    fname = filename.lower()
    if "__" in fname and "++" in fname:
        return fname.split("__", 1)[1].split("++", 1)[0]
    return ""

def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: Iterable[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

# ============================================================
# Part A: Gradle / GMD detection (logic copied unchanged)
# ============================================================

# --- Ignore folders commonly not relevant ---
SKIP_DIRS = {".git", ".idea", ".gradle", "build", "out", "node_modules", ".github", ".gitlab"}

# --- File filters ---
def is_any_gradle_file(path: str) -> bool:
    p = path.lower()
    return p.endswith(".gradle") or p.endswith(".gradle.kts")

# --- Comment stripping (handles // and /* */ but preserves http(s)://) ---
def strip_comments_gradle(text: str) -> str:
    if not text:
        return ""
    s = re.sub(r"/\*.*?\*/", "", text, flags=re.S)            # block comments
    s = re.sub(r"(?<!:)//.*?$", "", s, flags=re.M)            # line comments (not http://)
    return s

# --- Balanced block helper ---
def _find_block_span_from_head(text: str, head_start: int) -> Optional[Tuple[int, int]]:
    i = text.find("{", head_start)
    if i == -1:
        return None
    depth = 0
    for j in range(i, len(text)):
        ch = text[j]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return (i, j)
    return None

# convert char indexes to (start_line, end_line)
def _char_to_line_span(text: str, ci: Tuple[int, int]) -> Tuple[int, int]:
    a, b = ci
    start_line = text[:a].count("\n") + 1
    end_line   = text[:b+1].count("\n") + 1
    return start_line, end_line

# --- GMD field/name extractors ---
# Accept foo = "...", foo = '...', foo = identifier, and foo.set("...") / foo.set('...')
GMD_DEVICE_RX = re.compile(
    r'\bdevice\s*=\s*(?:"([^"]+)"|\'([^\']+)\'|([A-Za-z0-9_.]+))'
    r'|\bdevice\.set\(\s*(?:"([^"]+)"|\'([^\']+)\')\s*\)',
    re.I
)
GMD_API_RX = re.compile(
    r'\bapiLevel\s*=\s*"?(\d+)"?'
    r'|\bapiLevel\.set\(\s*"?(\d+)"?\s*\)',
    re.I
)
# allow unquoted identifiers for systemImageSource too
GMD_SRC_RX = re.compile(
    r'\bsystemImageSource\s*=\s*(?:"([^"]+)"|\'([^\']+)\'|([A-Za-z0-9_.-]+))'
    r'|\bsystemImageSource\.set\(\s*(?:"([^"]+)"|\'([^\']+)\')\s*\)',
    re.I
)

# Kotlin DSL: myPixel(ManagedVirtualDevice) { ... }   or fully qualified or ::class
GMD_BLOCK_NAME_RX = re.compile(
    r'(\w+)\s*\(\s*(?:com\.android\.build\.api\.dsl\.)?ManagedVirtualDevice(?:\s*::\s*class)?\s*\)',
    re.I
)
# Also support create("pixel6Api34", ManagedVirtualDevice[::class]) { ... }
GMD_CREATE_RX = re.compile(
    r'create\(\s*"([^"]+)"\s*,\s*(?:com\.android\.build\.api\.dsl\.)?ManagedVirtualDevice(?:\s*::\s*class)?\s*\)',
    re.I
)
# OPTIONAL: KTS generic create<ManagedVirtualDevice>("name")
GMD_CREATE_GENERIC_RX = re.compile(
    r'create\s*<\s*(?:com\.android\.build\.api\.dsl\.)?ManagedVirtualDevice\s*>\s*\(\s*"([^"]+)"\s*\)',
    re.I
)

# --- Token sets for quick prefiltering ---
PREFILTER_TOKENS = {
    # GMD-ish
    "manageddevices", "manageddevice", "managedvirtualdevice", "devicegroups", "devices", "groups", "testoptions",
    "alldevicesandroidtest", "pixel", "apilevel", "systemimagesource",
    # broader instrumentation config signals
    "androidtestimplementation", "androidtestapi", "androidtestruntimeonly", "androidtestcompileonly", "androidtestcompile",
    "testinstrumentationrunner", "testinstrumentationrunnerarguments", "enableandroidtest", "com.android.test",
    "androidtestutil", "androidx_test_orchestrator", "orchestrator"
}

# --- Find GMD blocks using a two-phase balanced approach ---
TESTOPTIONS_HEAD = re.compile(r"\btestOptions\s*\{", re.I)
MANAGED_HEAD     = re.compile(r"\bmanagedDevices\s*\{", re.I)
GMD_INNER_TOKENS_RX = re.compile(r"\b(managedDevices|managedVirtualDevice|devices|groups|deviceGroups)\b", re.I)

def find_gmd_line_spans(text: str) -> List[Tuple[int, int]]:
    """Return a list of (start_line, end_line) for candidate GMD-containing blocks."""
    spans: List[Tuple[int, int]] = []

    # 1) Direct managedDevices { ... } blocks
    for m in MANAGED_HEAD.finditer(text):
        ci = _find_block_span_from_head(text, m.start())
        if ci:
            spans.append(_char_to_line_span(text, ci))

    # 2) testOptions { ... } blocks that contain GMD tokens anywhere inside
    for m in TESTOPTIONS_HEAD.finditer(text):
        ci = _find_block_span_from_head(text, m.start())
        if not ci:
            continue
        a, b = ci
        sub = text[a:b+1]
        if GMD_INNER_TOKENS_RX.search(sub):
            spans.append(_char_to_line_span(text, (a, b)))

    return spans

def _first_group(m: re.Match) -> Optional[str]:
    if not m:
        return None
    for g in m.groups():
        if g:
            return g
    return None

def parse_gmd_block(text: str, span: Tuple[int, int]) -> Dict[str, Optional[str]]:
    lines = (text or "").splitlines()
    block_text = "\n".join(lines[span[0]-1:span[1]])
    norm: Dict[str, Optional[str]] = {
        "device_profile": None,
        "api_level": None,
        "image_source": None,
        "device_identifier": None,
        "context_anchor": None,
    }
    if (m := GMD_DEVICE_RX.search(block_text)):
        norm["device_profile"] = _first_group(m)
    if (m := GMD_API_RX.search(block_text)):
        norm["api_level"] = _first_group(m)
    if (m := GMD_SRC_RX.search(block_text)):
        norm["image_source"] = _first_group(m)
    if (m := GMD_BLOCK_NAME_RX.search(block_text)):
        ident = m.group(1)
        norm["device_identifier"] = ident
        norm["context_anchor"] = f"managedDevices.{ident}"
    elif (m := GMD_CREATE_RX.search(block_text)):
        ident = m.group(1)
        norm["device_identifier"] = ident
        norm["context_anchor"] = f"managedDevices.{ident}"
    elif (m := GMD_CREATE_GENERIC_RX.search(block_text)):
        ident = m.group(1)
        norm["device_identifier"] = ident
        norm["context_anchor"] = f"managedDevices.{ident}"
    return norm

V5_SIGNAL_TOKENS = [
    "androidtestimplementation", "androidtestapi", "androidtestcompileonly",
    "androidtestruntimeonly", "androidtestcompile",
    "testinstrumentationrunner", "testinstrumentationrunnerarguments",
    "androidtestutil",
    'execution "androidx_test_orchestrator"',
    "execution 'androidx_test_orchestrator'",
    'execution("ANDROIDX_TEST_ORCHESTRATOR")',
    "enableandroidtest",
    'id("com.android.test")', "id 'com.android.test'", 'apply plugin: "com.android.test"'
]

def has_instru_signal_config(text: str) -> bool:
    t = (text or "").lower()
    return any(tok in t for tok in V5_SIGNAL_TOKENS)

def walk_gradle_files(root: str) -> List[str]:
    out: List[str] = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
        for fname in filenames:
            if is_any_gradle_file(fname):
                out.append(os.path.join(dirpath, fname))
    return out

def prefilter_interesting(text: str) -> bool:
    t = (text or "").lower()
    return any(tok in t for tok in PREFILTER_TOKENS)

def scan_gradle_file(path: str) -> Dict[str, Any]:
    """Return dict with keys: text, v5_signal(bool), gmd_spans(list[span tuples]), gmd_detections(list[dict])."""
    try:
        raw = open(path, "r", encoding="utf-8", errors="ignore").read()
    except Exception:
        raw = ""
    text = strip_comments_gradle(raw)

    v5_signal = has_instru_signal_config(text)
    gmd_spans: List[Tuple[int, int]] = []
    gmd_detections: List[Dict[str, Optional[str]]] = []

    if prefilter_interesting(text):
        gmd_spans = find_gmd_line_spans(text)
        for sp in gmd_spans:
            det = parse_gmd_block(text, sp)
            det["span_start_line"], det["span_end_line"] = sp
            gmd_detections.append(det)

    return {
        "text": text,
        "v5_signal": v5_signal,
        "gmd_spans": gmd_spans,
        "gmd_detections": gmd_detections,
    }

def scan_all_gradle_files(root: Path) -> Tuple[List[Dict[str, Any]], Set[str]]:
    """
    Scan all Gradle files under CONFIG_DIR for GMD.
    Returns:
      gradle_rows: list of unified CSV rows for Gradle files
      gmd_repos: set of full_name (owner.repo) that have at least one GMD block
    """
    gradle_rows: List[Dict[str, Any]] = []
    gmd_repos: Set[str] = set()

    files = walk_gradle_files(str(root))
    for f in sorted(files):
        filename = os.path.basename(f)
        full_name = extract_full_name_from_file(filename)
        result = scan_gradle_file(f)
        gmd_detections = list(result["gmd_detections"])

        if gmd_detections:
            gmd_repos.add(full_name)
            for d in gmd_detections:
                gradle_rows.append({
                    "filename": filename,
                    "full_name": full_name,
                    "ci_platform": "",
                    "instru_t_ci_signal": False,
                    "execution_environment": "",
                    "test_invocation": "",
                    "flutter_integ_t_signal": False,
                    "flutter_integ_t_d": "",
                    "Exec_Env_Style": "GMD",
                    "Test_Invoc_Style": "",
                    "device_profile": d.get("device_profile"),
                    "api_level": d.get("api_level"),
                    "image_source": d.get("image_source"),
                    "device_identifier": d.get("device_identifier"),
                    "context_anchor": d.get("context_anchor"),
                    "span_start_line": d.get("span_start_line"),
                    "span_end_line": d.get("span_end_line"),
                })
        else:
            gradle_rows.append({
                "filename": filename,
                "full_name": full_name,
                "ci_platform": "",
                "instru_t_ci_signal": False,
                "execution_environment": "",
                "test_invocation": "",
                "flutter_integ_t_signal": False,
                "flutter_integ_t_d": "",
                "Exec_Env_Style": "",
                "Test_Invoc_Style": "",
                "device_profile": None,
                "api_level": None,
                "image_source": None,
                "device_identifier": None,
                "context_anchor": None,
                "span_start_line": None,
                "span_end_line": None,
            })

    return gradle_rows, gmd_repos

# ============================================================
# Part B: YAML / CI analysis (logic copied unchanged, output slimmed)
# ============================================================

COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')
def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

def normalize_block_keys(text: str) -> str:
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\||>|\|\-)\s*(.+)$', r'\2', text)
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(\||>|\|\-)\s*$', '', text)
    text = re.sub(
        r'(?m)^(\s*)-\s*(?=(?:\./|\.\\|bash|sh|pwsh|powershell|gradle(?:w)?|adb|flutter|gcloud|saucectl|appcenter)\b)',
        r'\1',
        text
    )
    return text

IGNORE_GHA_ACTIONS_RE = re.compile(
    r'(?mi)^\s*uses\s*:\s*('
    r'docker/(?:setup-qemu-action|setup-buildx-action|build-push-action|login-action)'
    r'|actions/checkout'
    r'|docker/setup-qemu-action'
    r'|docker/setup-buildx-action'
    r')@.*$'
)
def strip_irrelevant_ci_lines(text: str) -> str:
    return IGNORE_GHA_ACTIONS_RE.sub('', text or '')

# Gradle in YAML
GRADLE_PREFIX = (
    r'^\s*'
    r'(?:\S+=\S+\s+)*'
    r'(?:sudo\s+)?'
    r'(?:(?:bash|sh)\s+-c[l]?\s+[\'"]?)?'
    r'(?:[^#\n;]*?&&\s+)?'
    r'(?:cd\s+\S+\s+&&\s+)?'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?'
)
GRADLE_ANYWHERE_RE = re.compile(r'(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*')
GRADLE_BUILD_ACTION_RE = re.compile(r'(?mi)\buses\s*:\s*(gradle/gradle-build-action|gradle/actions/setup-gradle)@')

NON_TEST_PREFIX = r'(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)'
SHELL_PREFIX = r'(?:\S+=\S+\s+)*(?:sudo\s+)?(?:(?:bash|sh|pwsh|powershell)\s+-c\s+[\'"]?)?(?:[^#\n;]*?&&\s+)?'

EMULATOR_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}(?:\s|^)(?:(?:\./|\.\\)?(?:emulator)(?:\.exe)?)\b[^\n]*-avd\s+\S+'
ADB_WAIT_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}adb\s+wait[- ]?for[- ]?device\b'
ADB_SERIAL_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}adb\s+-s\s+(?:emulator-\d+|localhost:\d+|127\.0\.0\.1:\d+)'

DEVICE_SOURCES = [
    ("Real_Device", "adb -s <serial> (physical)", [
        r'(?m)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b'
    ]),
    ("Emulator", "adb -s emulator-serial", [ADB_SERIAL_LINE]),
    ("Emulator", "adb wait-for-device",   [ADB_WAIT_LINE]),
    ("Emulator", "emulator -avd/@",       [EMULATOR_LINE]),
    ("Emulator", "android-wait-for-emulator", [r'(?m)^\s*(?:\./)?android-wait-for-emulator\b']),
    ("Emulator", "start-emulator.sh", [r'(?m)^\s*start-emulator\.sh\b']),
    ("Emulator", "android create avd", [r'\bandroid\b[^\n]*\bcreate\s+avd\b']),
    ("Emulator", "circleci android orb", [
        r"(?mi)^\s*(?:-\s*)?android/(?:start-emulator-and-run-tests|create-avd|launch-emulator)\s*:",
        r"(?mi)^\s*system-image\s*:\s*system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*);(?:default|google_apis)[^\s]*"
    ]),
    ("Emulator", "reactivecircus runner", [r'(?mi)\buses\s*:\s*reactivecircus/android-emulator-runner@[\w\.\-]+']),
    ("Emulator", "malinskiy runner", [r'(?mi)\buses\s*:\s*malinskiy/action-android/emulator-run-cmd@[\w\.\-]+']),
    ("Emulator", "sys-img component", [
        r'(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
        r'(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    ("Emulator", "avdmanager", [r'(?m)^\s*\S*avdmanager\b']),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"',
        r'^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
    ]),
    # Weak hints (filtered if no strong signals)
    ("Emulator", "headless flag", [r'(?mi)\b-no-?audio\b', r'(?mi)\b-no-window\b', r'(?mi)\b-no-boot-anim\b']),
    ("Emulator", "avd-name", [r'(?mi)^\s*avd[-_ ]?name\s*:\s*\S+']),
    ("Emulator", "api-level", [r'(?mi)\bapi[-_ ]?level\s*:\s*\d{2,}|\bapi_level\s*:\s*\d{2,}']),
    ("Emulator", "abi/arch", [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image", [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "device name", [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),
    # Third-party labs (environment)
    ("Third_Party_Lab", "gcloud firebase", [r'(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",        [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack", [r'(?i)\b(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test",  [r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "maestro cloud",   [r'(?mi)^[^\n]*\bmaestro\s+cloud\b']),
    ("Third_Party_Lab", "emulator.wtf action", [
        r'(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+',
        r'(?i)\bemulator\.wtf\b'
    ]),
    # Other emulator actions
    ("Emulator", "other gha emulator", [
        r'(?mi)^\s*uses\s*:\s*vgaidarji/android-github-actions-emulator@[\w\.\-]+',
        r'(?mi)^\s*uses\s*:\s*(?!reactivecircus/android-emulator-runner@)'
        r'(?!malinskiy/action-android/emulator-run-cmd@)'
        r'(?!emulator-wtf/run-tests@)'
        r'[\w\.-]+/[\w\./-]*android[\w\./-]*(?:\bemulator\b|\bavd\b)[\w\./-]*@[\w\.\-]+'
    ]),
]
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]

EXCLUDED_TASK_SEGMENT_RE = re.compile(
    r'(^|\s)(?:-x|--exclude-task)\s+(["\']?)[:\w\.-]*(?:androidtest|baselineprofile)[\w:\.-]*\2\b',
    re.IGNORECASE | re.MULTILINE,
)
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")
def remove_excluded_gradle_tasks(text: str) -> str:
    return EXCLUDED_TASK_SEGMENT_RE.sub(lambda m: (m.group(1) or " "), text or "")
def pre_sanitize(text: str) -> str:
    t = remove_excluded_gradle_tasks(text)
    return GHA_EXPR_RE.sub("", t or "")

TRIGGER_SOURCES_PRIMARY = [
    ("Gradle",  "connectedAndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedandroidtest\b[^\n\r]*']),
    ("Gradle", "connected.*Android.*", [
        rf'''(?mix){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b(?![^\n\r]*\b(?:{NON_TEST_PREFIX})[\w-]*androidtest\b)[^\n\r]*'''
    ]),
    ("Gradle",  "connectedCheck",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedcheck\b[^\n\r]*']),
    ("Gradle",  "cAT shorthand",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*cAT\b[^\n\r]*']),
    ("Gradle",  "deviceCheck",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b[^\n\r]*']),
    ("Gradle",  "managedDevice AndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b[^\n\r]*']),
    ("Gradle", "variant/device AndroidTest",
     [rf'''(?mix){GRADLE_PREFIX}[^\n\r]*\b(?:(?:[:\w-]+:)*(?!{NON_TEST_PREFIX})(?!connected)(?!spoon)(?!marathon)[A-Za-z0-9][\w-]*androidtest\b)[^\n\r]*''']),
    ("Gradle",  "Spoon",    [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\bspoon(?:\w*androidtest)?\b']),
    ("Gradle",  "Marathon", [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\bmarathon(?:\w*androidtest)?\b']),
    ("ADB",     "am instrument", [r'(?mi)^[^\n]*\bam\s+instrument\b']),
    ("Third_Party_Lab", "gcloud firebase (instr)", [r'(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)']),
    ("Third_Party_Lab", "flank",            [r'(?mi)^[^\n]*\bflank\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",         [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run",    [r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "emulator.wtf run", [
        r'(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+',
        r'(?i)\bemulator\.wtf\b'
    ]),
    ("Gradle", "generateBaselineProfile", [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*generate(?:\w*?)baselineprofile\b[^\n\r]*']),
    ("Gradle", "collectBaselineProfile",  [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*collect(?:\w*?)baselineprofile\b[^\n\r]*']),
    ("Gradle", "connectedBenchmarkAndroidTest", [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedbenchmarkandroidtest\b[^\n\r]*']),
]
TRIGGER_SOURCES_ANYWHERE = [
    ("Gradle", "connected (anywhere)", [
        r'(?mix)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b'
    ]),
    ("Gradle", "connectedAndroidTest (anywhere)", [r'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bconnectedandroidtest\b']),
    ("Gradle", "connectedCheck (anywhere)",       [r'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\b(?:[:\w-]+:)*connectedcheck\b']),
    ("Gradle", "managedDevice AndroidTest (anywhere)", [
        rf'(?mix)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\b(?:(?:[:\w-]+:)*(?!{NON_TEST_PREFIX})(?!connected)(?!spoon)(?!marathon)[A-Za-z0-9][\w-]*androidtest\b)'
    ]),
    ("Gradle", "variant/device AndroidTest (anywhere)", [
        rf'(?mix)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\b(?:(?:[:\w-]+:)*(?!{NON_TEST_PREFIX})[\w-]*androidtest\b)'
    ]),
    ("Gradle", "Spoon (anywhere)",   [r'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bspoon(?:\w*androidtest)?\b']),
    ("Gradle", "Marathon (anywhere)",[r'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bmarathon(?:\w*androidtest)?\b']),
    ("Gradle", "generateBaselineProfile (anywhere)", [rf'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bgenerate(?:\w*?)baselineprofile\b']),
    ("Gradle", "collectBaselineProfile (anywhere)",  [rf'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bcollect(?:\w*?)baselineprofile\b']),
    ("Gradle", "connectedBenchmarkAndroidTest (anywhere)", [rf'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bconnectedbenchmarkandroidtest\b']),
]
TRIGGER_PATTERNS_PRIMARY  = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_PRIMARY]
TRIGGER_PATTERNS_ANYWHERE = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_ANYWHERE]

GHA_GRADLE_INPUTS = compile_any([
    rf'''(?mix)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connected(?:\${{\s*[^}}]+\s*}}|[^\n\r])*?android(?:\${{\s*[^}}]+\s*}}|[^\n\r])*?test\b''',
    r'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    rf'''(?mix)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:(?:[:\w-]+:)*(?!{NON_TEST_PREFIX})[\w-]*androidtest\b)''',
    r'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*cAT\b',
])
GHA_GMD_INPUTS = compile_any([
    rf'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?!(?:[:\w-]+:)*(?:connected[a-z0-9:._-]*|{NON_TEST_PREFIX})[\w:-]*androidtest\b)(?:[:\w-]+:)*[\w:-]*androidtest\b'
])

FLUTTER_IT_LINE = re.compile(r'(?mi)^\s*flutter\s+(?:test|drive)\b[^\n]*')
FLUTTER_IT_ANDROID_HINT = re.compile(r'(?i)(integration_test|--driver\b|/integration_test/)')
FLUTTER_DEVICE_FLAG_RE = re.compile(r'(?i)\s+-d\s+(?P<dev>"[^"]+"|\'[^\']+\'|\S+)')
FLUTTER_DEVICE_IS_ANDROID = re.compile(r'(?i)\b(android|emulator-\d+|sdk\s+gphone|android\s+sdk\s+built\s+for|pixel)\b')
LINUX_HEADLESS_HINTS_RE = re.compile(r'(?mi)^\s*(xvfb-run|export\s+DISPLAY=|sudo\s+Xvfb)\b')

PROVIDER_PATTERNS = [
    ("emulator-wtf", compile_any([r'(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@', r'(?i)\bemulator\.wtf\b'])),
    ("firebase-test-lab", compile_any([r'(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b', r'(?mi)\bflank\s+android\s+run\b'])),
    ("browserstack", compile_any([r'(?i)\bbrowserstack\b', r'(?i)\bbstack\b'])),
    ("aws-device-farm", compile_any([r'(?mi)\baws\s+devicefarm\b'])),
    ("sauce-labs", compile_any([r'(?mi)\bsaucectl(?:\s+run)?\b', r'(?mi)\bsauce\s+ctl\b'])),
    ("appcenter", compile_any([r'(?mi)\bappcenter\s+test\s+run\s+android\b'])),
    ("maestro-cloud", compile_any([r'(?mi)\bmaestro\s+cloud\b'])),
]
INLINE_DEVICE_HINTS = compile_any([
    r'(?mi)^\s*devices\s*:\s*\|',
    r'(?mi)\b--device\b',
    r'(?mi)\bmodel\s*=\s*[^,\s]+',
    r'(?mi)\bversion\s*=\s*\d+',
    r'(?mi)\blocale\s*=\s*[-\w]+',
    r'(?mi)\borientation\s*=\s*(portrait|landscape)',
    r'(?mi)^\s*with-orchestrator\s*:\s*true\b',
    r'(?mi)\b--use-orchestrator\b',
    r'(?mi)\bnum-flaky-test-attempts\s*:\s*\d+\b',
    r'(?mi)\b--num-flaky-test-attempts(?:=|\s+)\d+\b',
])
FTL_HAS_INSTRUMENTATION = re.compile(r'(?mi)\bgcloud\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)')
FTL_APP_ONLY = re.compile(r'(?mi)\bgcloud\s+firebase\s+test\s+android\s+run\b(?![^\n]*\b(--test\b|--type\s+instrumentation\b))')
CONFIG_FILE_HINTS = compile_any([
    r'(?mi)\.ewtf\.ya?ml\b', r'(?mi)\b(flank\.ya?ml|flank\.android\.ya?ml)\b',
    r'(?mi)\b--config(?:=|\s+)\S+', r'(?mi)\b(browserstack\.ya?ml)\b', r'(?mi)\b(bs(?:config)?\.ya?ml)\b',
])
ORCHESTRATOR_HINTS = compile_any([r'(?mi)^\s*with-orchestrator\s*:\s*true\b', r'(?mi)\b--use-orchestrator\b'])
RETRY_HINTS = compile_any([r'(?mi)\bnum-flaky-test-attempts\s*:\s*(\d+)\b', r'(?mi)\b--num-flaky-test-attempts(?:=|\s+)(\d+)\b'])

def detect_provider(text: str) -> str:
    for name, pats in PROVIDER_PATTERNS:
        if any_match(pats, text):
            return name
    return "other"

def detect_inline_env(text: str) -> bool:
    return any_match(INLINE_DEVICE_HINTS, text)

def detect_config_env(text: str) -> bool:
    return any_match(CONFIG_FILE_HINTS, text)

def count_devices(text: str) -> int:
    count = 0
    m = re.search(r'(?mi)^\s*devices\s*:\s*\|\s*([\s\S]+)', text)
    if m:
        block = m.group(1)
        lines = [ln for ln in block.splitlines() if ln.strip()]
        pruned = []
        for ln in lines:
            if re.match(r'^\s*\w[\w-]*\s*:\s*', ln): break
            pruned.append(ln)
        count += sum(1 for ln in pruned if re.search(r'\bmodel\s*=', ln))
    count += len(re.findall(r'(?mi)\b--device\b', text))
    return count or 0

def detect_orchestrator(text: str) -> bool:
    return any_match(ORCHESTRATOR_HINTS, text)

def detect_retries(text: str) -> int:
    for pat in RETRY_HINTS:
        m = pat.search(text)
        if m:
            try: return int(m.group(1))
            except Exception: pass
    return 0

EMULATOR_STRONG_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner", "avdmanager",
    "sdkmanager system-images/emulator", "adb -s emulator-serial", "adb wait-for-device",
    "android create avd", "malinskiy runner", "other gha emulator", "circleci android orb",
}
THIRD_PARTY_STRONG_LABELS = {
    "gcloud firebase", "emulator.wtf action", "saucectl",
    "browserstack/bstack", "appcenter test", "maestro cloud"
}
REAL_DEVICE_STRONG_LABELS = {"adb -s <serial> (physical)"}
REAL_DEVICE_GENERIC_ADB = set()
STRONG_DEVICE_LABELS = EMULATOR_STRONG_LABELS | REAL_DEVICE_STRONG_LABELS | THIRD_PARTY_STRONG_LABELS

def filter_weak_device_hints(labels, groups):
    lbl_set = set(labels)
    if not (lbl_set & STRONG_DEVICE_LABELS):
        labels = [l for l in labels if l in STRONG_DEVICE_LABELS]
        if not labels: groups = []
    return labels, groups

def reconcile_emulator_vs_real(labels, groups):
    lbls = set(labels)
    if lbls & EMULATOR_STRONG_LABELS:
        lbls -= REAL_DEVICE_GENERIC_ADB
        labels = [l for l in labels if l in lbls]
        if "Real_Device" in groups:
            has_real_after = bool(set(labels) & REAL_DEVICE_STRONG_LABELS)
            if not has_real_after:
                groups = [g for g in groups if g != "Real_Device"]
    return labels, groups

ANDROID_CONTEXT_RE = re.compile(
    r'(?i)\b(adb|avd|emulator|android\s+sdk|system-images;android-|androidtest|connected(check|androidtest)|gcloud\s+firebase\s+test\s+android\s+run)\b'
)

JOBS_ANCHOR_RE = re.compile(r'(?m)^(?P<indent>\s*)jobs\s*:\s*$')
ANY_KEY_RE     = re.compile(r'(?m)^(?P<indent>\s*)(?P<name>[\w-]+)\s*:\s*$')

def split_jobs_blocks(raw: str) -> List[Tuple[str, str]]:
    m = JOBS_ANCHOR_RE.search(raw)
    if not m:
        return [("__whole__", raw)]
    jobs_indent = len(m.group("indent"))
    lines = raw.splitlines(True)
    start_idx = raw[:m.end()].count("\n")
    candidates = []
    for i in range(start_idx, len(lines)):
        lm = ANY_KEY_RE.match(lines[i])
        if not lm:
            continue
        indent = len(lm.group("indent"))
        if indent > jobs_indent:
            candidates.append((i, indent, lm.group("name")))
    if not candidates:
        return [("__whole__", raw)]
    min_indent = min(indent for _, indent, _ in candidates)
    job_headers = [(i, name) for (i, indent, name) in candidates if indent == min_indent]
    if not job_headers:
        return [("__whole__", raw)]
    blocks = []
    header_indices = [i for i, _ in job_headers] + [len(lines)]
    for idx in range(len(job_headers)):
        i, name = job_headers[idx]
        j = header_indices[idx + 1]
        block_text = "".join(lines[i:j])
        blocks.append((name, block_text))
    return blocks

def collect_hits_with_groups(patterns: List[Tuple[str, str, List[Pattern]]], text: str):
    labels, groups = [], []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl)
            groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

# ====== Exec/Test style derivation ======

def derive_exec_env_style(exec_envs: List[str], full_name: str, gmd_repos: Set[str]) -> str:
    """
    Map fine-grained execution_environment to coarse Exec_Env_Style:
      - Emu_Community  (ReactiveCircus / Malinskiy runners)
      - Emu_Custom     (Emulator_DIY / Emulator_Other)
      - Third-Party
      - Real Device
      - GMD           (repo has any GMD Gradle block)
    """
    styles: Set[str] = set()
    envs = set(exec_envs)

    if "Emulator_ReactiveCircus" in envs or "Emulator_Malinskiy" in envs:
        styles.add("Emu_Community")

    if "Emulator_Other" in envs or "Emulator_DIY" in envs:
        styles.add("Emu_Custom")

    if "Third Party" in envs:
        styles.add("Third-Party")

    if "Real Device" in envs:
        styles.add("Real Device")

    if full_name in gmd_repos:
        styles.add("GMD")

    return ",".join(sorted(styles)) if styles else ""

def derive_test_invoc_style(test_inv: str) -> str:
    """
    Map fine-grained test_invocation to coarse Test_Invoc_Style:
      - Gradle-based   (any Gradle_* invocation)
      - Third-Party CLI (3P CLIs)
      - ADB            (am instrument / adb invocations)
    """
    styles: Set[str] = set()
    tokens = {t.strip() for t in (test_inv or "").split(",") if t.strip()}

    if any(t.startswith("Gradle") for t in tokens):
        styles.add("Gradle-based")
    if "3P CLIs" in tokens:
        styles.add("Third-Party CLI")
    if "ADB" in tokens:
        styles.add("ADB")

    return ",".join(sorted(styles)) if styles else ""

# === map invocations & envs (unchanged logic) ===

def has_gmd_gradle_trigger(trigger_labels: List[str]) -> bool:
    L = {l.lower() for l in trigger_labels}
    if any("manageddevice androidtest" in l for l in L): return True
    if ("variant/device androidtest" in L and not any(x in L for x in {
        "connected.*android.*","connectedandroidtest","connectedbenchmarkandroidtest","spoon","marathon"
    })): return True
    return False

def has_connected_gradle_trigger(trigger_labels: List[str]) -> bool:
    L = {l.lower() for l in trigger_labels}
    keys = {"connected.*android.*","connectedandroidtest","connectedbenchmarkandroidtest","connectedcheck",
            "cat shorthand","connected (anywhere)","connectedandroidtest (anywhere)",
            "connectedcheck (anywhere)","spoon","marathon","devicecheck","gha gradle inputs/script"}
    return any(k in L for k in keys) or any(("connected" in l and "android" in l and "test" in l) for l in L)

def has_baselineprofile_trigger(trigger_labels: List[str]) -> bool:
    L = {l.lower() for l in trigger_labels}
    return any("baselineprofile" in l for l in L)

def map_test_invocations(groups: List[str], trigger_labels: List[str], file_flutter_it_androidish: bool) -> List[str]:
    s_groups = set(groups)
    L = {l.lower() for l in trigger_labels}
    out: List[str] = []
    if has_gmd_gradle_trigger(trigger_labels): out.append("Gradle_GMD")
    if has_connected_gradle_trigger(trigger_labels): out.append("Gradle_Connected")
    if has_baselineprofile_trigger(trigger_labels): out.append("Gradle")
    if "ADB" in s_groups: out.append("ADB")
    if "Third_Party_Lab" in s_groups: out.append("3P CLIs")

    has_gradle_or_adb = any(x in out for x in ["Gradle_GMD","Gradle_Connected","Gradle","ADB"])
    if (not has_gradle_or_adb) and ("flutter integration test" in L) and file_flutter_it_androidish:
        out.append("3P CLIs")

    return sorted(set(out))

def map_execution_envs(groups: List[str], labels: List[str]) -> List[str]:
    envs = set(); s_groups, s_labels = set(groups), set(labels)
    if "Third_Party_Lab" in s_groups: envs.add("Third Party")
    if "Real_Device" in s_groups: envs.add("Real Device")
    if "Emulator" in s_groups:
        if "reactivecircus runner" in s_labels: envs.add("Emulator_ReactiveCircus")
        elif "malinskiy runner" in s_labels: envs.add("Emulator_Malinskiy")
        elif ("other gha emulator" in s_labels) or ("circleci android orb" in s_labels): envs.add("Emulator_Other")
        else: envs.add("Emulator_DIY")
    return sorted(envs)

# ============================================================
# main
# ============================================================

def main():
    # 1) Gradle / GMD pass
    gradle_rows, gmd_repos = scan_all_gradle_files(CONFIG_DIR)

    # 2) YAML / CI pass
    yaml_rows: List[Dict[str, Any]] = []

    for f in sorted(CONFIG_DIR.iterdir()):
        if not f.is_file():
            continue
        if f.suffix.lower() not in (".yml", ".yaml"):
            continue

        filename = f.name
        full_name = extract_full_name_from_file(filename)
        ci_platform = extract_ci_platform(filename)

        try:
            raw = f.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            raw = ""

        file_trigger_labels: List[str] = []
        file_device_labels:  List[str] = []
        file_trigger_groups: List[str] = []
        file_device_groups:  List[str] = []
        file_has_test_trigger = False
        file_has_device_with_group = False
        file_gradle_present_any = False
        file_instru_signal_any = False
        file_flutter_devices: List[str] = []

        file_flutter_it_present = False
        file_flutter_it_androidish = False

        file_provider: str = "other"
        file_env_declared = False
        file_env_location = "none"   # none | inline | config
        file_device_count = 0
        file_orchestrator = False
        file_retries = 0

        for job_name, job_raw in split_jobs_blocks(raw):
            content = strip_comments(job_raw)
            content = normalize_block_keys(content)
            content = strip_irrelevant_ci_lines(content)
            content_for_triggers = pre_sanitize(content)

            prov = detect_provider(content_for_triggers)
            if prov != "other":
                file_provider = prov
            inline_decl = detect_inline_env(content_for_triggers)
            config_decl = detect_config_env(content_for_triggers)
            if inline_decl:
                file_env_declared = True
                file_env_location = "inline"
            elif (not file_env_declared) and config_decl:
                file_env_declared = True
                file_env_location = "config"
            file_device_count += count_devices(content_for_triggers)
            if detect_orchestrator(content_for_triggers):
                file_orchestrator = True
            file_retries = max(file_retries, detect_retries(content_for_triggers))

            dev_labels, dev_groups = collect_hits_with_groups(DEVICE_PATTERNS, content_for_triggers.lower())
            dev_labels, dev_groups = filter_weak_device_hints(dev_labels, dev_groups)
            dev_labels, dev_groups = reconcile_emulator_vs_real(dev_labels, dev_groups)

            trig_labels, trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, content_for_triggers.lower())

            if any_match(GHA_GRADLE_INPUTS, content_for_triggers):
                has_gradle_anywhere = bool(GRADLE_ANYWHERE_RE.search(content_for_triggers) or GRADLE_BUILD_ACTION_RE.search(content_for_triggers))
                tied_to_emulator_action = bool(re.search(
                    r'(?mi)^\s*uses\s*:\s*(?:reactivecircus/android-emulator-runner|malinskiy/action-android/emulator-run-cmd|hannesa2/action-android/emulator-run-cmd)\@',
                    content_for_triggers
                ))
                if has_gradle_anywhere or tied_to_emulator_action:
                    trig_labels = unique_preserve(trig_labels + ["gha gradle inputs/script"])
                    trig_groups = unique_preserve(trig_groups + ["Gradle"])

            if any_match(GHA_GMD_INPUTS, content_for_triggers):
                trig_labels = unique_preserve(trig_labels + ["variant/device AndroidTest"])
                trig_groups = unique_preserve(trig_groups + ["Gradle"])

            flutter_it_hits = []
            for m in FLUTTER_IT_LINE.finditer(content_for_triggers):
                line = m.group(0)
                if FLUTTER_IT_ANDROID_HINT.search(line) or FLUTTER_DEVICE_IS_ANDROID.search(line):
                    flutter_it_hits.append(line)

            flutter_it_android_targeted = False
            if flutter_it_hits:
                trig_labels = unique_preserve(trig_labels + ["flutter integration test"])
                trig_groups = unique_preserve(trig_groups + ["Flutter"])
                file_flutter_it_present = True

                for line in flutter_it_hits:
                    d = FLUTTER_DEVICE_FLAG_RE.search(line)
                    if d:
                        plat = d.group("dev").strip('"\'').lower()
                        if (plat == "android" or plat.startswith("emulator-")
                            or "sdk gphone" in plat or "android sdk built for" in plat or "pixel" in plat):
                            flutter_it_android_targeted = True
                            if "android" not in file_flutter_devices: file_flutter_devices.append("android")
                        elif plat in {"linux","macos","windows"}:
                            if plat not in file_flutter_devices: file_flutter_devices.append(plat)
                        elif plat in {"ios","iphone","ipad","iphone simulator"}:
                            if "ios" not in file_flutter_devices: file_flutter_devices.append("ios")
                        elif plat in {"web","web-server","chrome","edge","firefox","safari"}:
                            if "web" not in file_flutter_devices: file_flutter_devices.append("web")
                if not file_flutter_devices and LINUX_HEADLESS_HINTS_RE.search(content_for_triggers):
                    if "linux" not in file_flutter_devices: file_flutter_devices.append("linux")

            fb_trig_labels, fb_trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_ANYWHERE, content_for_triggers.lower())
            if fb_trig_labels:
                trig_labels = unique_preserve(trig_labels + fb_trig_labels)
                trig_groups = unique_preserve(trig_groups + fb_trig_groups)

            has_test_trigger = bool(trig_labels)
            if "Third_Party_Lab" in dev_groups:
                inline = detect_inline_env(content_for_triggers)
                cfgref = detect_config_env(content_for_triggers)
                has_ftl_instr = bool(FTL_HAS_INSTRUMENTATION.search(content_for_triggers))
                bs_test_hint = re.search(
                    r'(?mi)\b(browserstack|bstack)\b[^\n]*\b(espresso|instrumentation|app-automate|automate|--device|--devices)\b',
                    content_for_triggers
                )
                if (not has_test_trigger) and (not inline) and (not cfgref) and (not has_ftl_instr) and (not bs_test_hint):
                    dev_labels = [l for l in dev_labels if l not in {
                        "browserstack/bstack","gcloud firebase","saucectl","appcenter test","maestro cloud","emulator.wtf action"
                    }]
                    if not any(l in {"browserstack/bstack","gcloud firebase","saucectl","appcenter test","maestro cloud","emulator.wtf action"} for l in dev_labels):
                        dev_groups = [g for g in dev_groups if g != "Third_Party_Lab"]

            android_context = bool(ANDROID_CONTEXT_RE.search(content_for_triggers))
            if ("other gha emulator" in dev_labels) and (not android_context) and (not has_test_trigger):
                dev_labels = [l for l in dev_labels if l != "other gha emulator"]
                if not dev_labels:
                    dev_groups = [g for g in dev_groups if g != "Emulator"]

            ANDROID_SPECIFIC_3P_LABELS = {"gcloud firebase", "appcenter test", "emulator.wtf action", "saucectl"}
            has_android_env = (
                any(g in {"Emulator", "Real_Device"} for g in dev_groups) or
                any(lbl in ANDROID_SPECIFIC_3P_LABELS for lbl in dev_labels)
            )
            if flutter_it_hits and (flutter_it_android_targeted or has_android_env):
                file_flutter_it_androidish = True

            has_device_setup = bool(dev_labels)
            gradle_present   = bool(GRADLE_ANYWHERE_RE.search(content) or GRADLE_BUILD_ACTION_RE.search(content))

            file_device_labels  = unique_preserve(file_device_labels  + dev_labels)
            file_device_groups  = unique_preserve(file_device_groups  + dev_groups)
            file_trigger_labels = unique_preserve(file_trigger_labels + trig_labels)
            file_trigger_groups = unique_preserve(file_trigger_groups + trig_groups)
            file_has_test_trigger = file_has_test_trigger or bool(trig_labels)
            if has_device_setup and any(g in {"Emulator","Third_Party_Lab","Real_Device"} for g in dev_groups):
                file_has_device_with_group = True
            file_gradle_present_any = file_gradle_present_any or gradle_present

            non_flutter_triggers = [l for l in trig_labels if l.lower() != "flutter integration test"]
            flutter_counts_as_instr = (("flutter integration test" in [l.lower() for l in trig_labels])
                                       and (flutter_it_android_targeted or has_android_env))

            instru_t_ci_signal_job = bool(
                non_flutter_triggers or
                flutter_counts_as_instr or
                (has_device_setup and any(g in {"Emulator","Third_Party_Lab","Real_Device"} for g in dev_groups))
            )
            file_instru_signal_any = file_instru_signal_any or instru_t_ci_signal_job

        # map invocations & envs
        combined_groups = unique_preserve(file_trigger_groups + file_device_groups)
        test_inv_list = map_test_invocations(combined_groups, file_trigger_labels, file_flutter_it_androidish)
        test_inv = ",".join(test_inv_list)
        exec_envs = map_execution_envs(file_device_groups, file_device_labels)

        exec_env_style = derive_exec_env_style(exec_envs, full_name, gmd_repos)
        test_inv_style = derive_test_invoc_style(test_inv)

        third_party_label = ""
        if "Third Party" in exec_envs:
            if file_env_declared and file_env_location == "inline":
                third_party_label = "Third-Party Lab — Explicit Inline Env"
            elif file_env_declared and file_env_location == "config":
                third_party_label = "Third-Party Lab — Config-Referenced Env"
            else:
                third_party_label = "Third-Party Lab — Invocation Only"

        csv_row = {
            "filename": filename,
            "full_name": full_name,
            "ci_platform": ci_platform,
            "instru_t_ci_signal": bool(file_instru_signal_any),
            "execution_environment": ",".join(exec_envs),
            "test_invocation": test_inv,
            "flutter_integ_t_signal": bool(file_flutter_it_present),
            "flutter_integ_t_d": ",".join(sorted(set(file_flutter_devices))),
            "Exec_Env_Style": exec_env_style,
            "Test_Invoc_Style": test_inv_style,
            # GMD detail columns (YAML rows → blank)
            "device_profile": None,
            "api_level": None,
            "image_source": None,
            "device_identifier": None,
            "context_anchor": None,
            "span_start_line": None,
            "span_end_line": None,
        }
        yaml_rows.append(csv_row)

    # 3) Combine YAML + Gradle rows and save
    all_rows = yaml_rows + gradle_rows
    out_df = pd.DataFrame(all_rows)
    out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved unified config CSV: {OUTPUT_CSV} (rows={len(all_rows)})")

if __name__ == "__main__":
    main()
